# Climate Debt Analysis: Bangladesh Master Pipeline

This notebook executes a comprehensive 3-stage data analysis pipeline connecting **Bangladesh's subnational rainfall anomalies**, **food price volatility**, and **macro-level external debt**.

### Pipeline Structure:
1. **Stage 1: Individual EDA** - Baseline patterns and clustering.
2. **Stage 2: Pairwise Alignment** - Cross-pillar correlations and causality.
3. **Stage 3: Joint Mediation** - The causal mechanism (Rainfall -> Food -> Debt).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests
import os

# Settings
sns.set(style="whitegrid")
os.makedirs('production_artifacts/figures', exist_ok=True)

PCODE_MAP = {
    'BD10': 'Barisal', 'BD20': 'Chittagong', 'BD30': 'Dhaka', 
    'BD40': 'Khulna', 'BD50': 'Rajshahi', 'BD55': 'Rangpur', 
    'BD60': 'Sylhet', 'BD45': 'Mymensingh'
}

## Stage 1: Individual Dataset EDA
Analyzing rainfall, food prices, and debt independently.

In [ ]:
# Load Datasets
rf_df = pd.read_csv('datasets/bgd-rainfall-subnat-full.csv')
food_df = pd.read_csv('datasets/wfp_food_prices_bgd.csv')
debt_df = pd.read_csv('datasets/external-debt_bgd.csv')

rf_df['date'] = pd.to_datetime(rf_df['date'])
food_df['date'] = pd.to_datetime(food_df['date'])

print("Datasets Loaded.")

### 1.1 Rainfall Clusters
Grouping regions by climate risk profile.

In [ ]:
region_stats = rf_df.groupby('PCODE')['rfq'].agg(['mean', 'std']).reset_index()
scaler = StandardScaler()
X_rf_cluster = scaler.fit_transform(region_stats[['mean', 'std']])
region_stats['cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_rf_cluster)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=region_stats, x='mean', y='std', hue='cluster', palette='viridis', s=100)
plt.title('Rainfall Clusters: Regional Mean Anomaly vs Variability')
plt.show()

### 1.2 Food Price Trends
Forecasting national rice prices as a baseline.

In [ ]:
rice_df = food_df[food_df['commodity'] == 'Rice (paddy)'].groupby('date')['price'].mean().reset_index()
rice_df['year'] = rice_df['date'].dt.year
rice_df['month'] = rice_df['date'].dt.month

gb_model = GradientBoostingRegressor(n_estimators=100)
gb_model.fit(rice_df[['year', 'month']], rice_df['price'])
rice_df['pred'] = gb_model.predict(rice_df[['year', 'month']])

plt.figure(figsize=(12, 5))
plt.plot(rice_df['date'], rice_df['price'], label='Actual')
plt.plot(rice_df['date'], rice_df['pred'], label='Forecast', linestyle='--')
plt.title('Rice Price Trends & GB Baseline Forecast')
plt.legend()
plt.show()

### 1.3 External Debt Drivers
Identifying macro factors correlated with debt stock.

In [ ]:
relevant_indicators = {
    'External debt stocks, total (DOD, current US$)': 'total_debt',
    'Imports of goods, services and primary income (BoP, current US$)': 'imports',
    'Exports of goods, services and primary income (BoP, current US$)': 'exports',
    'Personal remittances, received (current US$)': 'remittances'
}
debt_filtered = debt_df[debt_df['Indicator Name'].isin(relevant_indicators.keys())].copy()
debt_filtered['Indicator Name'] = debt_filtered['Indicator Name'].map(relevant_indicators)
wide_debt = debt_filtered.pivot_table(index='Year', columns='Indicator Name', values='Value').dropna()

X_debt = scaler.fit_transform(wide_debt.drop(columns='total_debt'))
y_debt = wide_debt['total_debt']
ridge = Ridge(alpha=1.0).fit(X_debt, y_debt)

plt.figure(figsize=(8, 4))
pd.Series(ridge.coef_, index=wide_debt.drop(columns='total_debt').columns).plot(kind='barh')
plt.title('Debt Driver Coefficients (Ridge Regression)')
plt.show()

## Stage 2: Pairwise Alignment
Linking variables to understand cross-pillar impacts.

In [ ]:
# Rainfall vs Food Price (Subnational)
rf_agg = rf_df.groupby(['date', 'admin1'])['rfq'].mean().reset_index()
food_agg = food_df.groupby(['date', 'admin1'])['price'].mean().reset_index()
merged_ab = pd.merge(rf_agg, food_agg, on=['date', 'admin1'])

plt.figure(figsize=(8, 6))
sns.regplot(data=merged_ab, x='rfq', y='price', scatter_kws={'alpha':0.2})
plt.title('Rainfall Anomaly vs Food Price (Subnational Alignment)')
plt.show()

## Stage 3: Joint Mediation Analysis
Testing the chain: **Rainfall -> Food Prices -> External Debt**.

In [ ]:
# Construct Master Dataset
rf_annual = rf_df.groupby(rf_df['date'].dt.year)['rfq'].agg(['mean', 'std']).reset_index()
rf_annual.columns = ['year', 'rain_anomaly', 'rain_var']
food_annual = food_df.groupby(food_df['date'].dt.year)['price'].agg(['mean', 'std']).reset_index()
food_annual.columns = ['year', 'food_price', 'food_volatility']
debt_growth = wide_debt['total_debt'].pct_change().reset_index().rename(columns={'Year':'year', 'total_debt':'debt_growth'})

master_df = pd.merge(rf_annual, food_annual, on='year')
master_df = pd.merge(master_df, debt_growth, on='year').dropna()

# Mediation Test
X = sm.add_constant(master_df['rain_anomaly'])
M = master_df['food_volatility']
Y = master_df['debt_growth']

model_total = sm.OLS(Y, X).fit()
model_path_a = sm.OLS(M, X).fit()
model_full = sm.OLS(Y, sm.add_constant(pd.concat([master_df['rain_anomaly'], M], axis=1))).fit()

print("--- MEDIATION RESULTS ---")
print(f"Rain -> Debt (Total Effect) P-val: {model_total.pvalues['rain_anomaly']:.4f}")
print(f"Rain -> Food (Path A) P-val: {model_path_a.pvalues['rain_anomaly']:.4f}")
print(f"Food -> Debt (Path B) P-val: {model_full.pvalues['food_volatility']:.4f}")

### Risk Era Visualization
Categorizing years into clusters based on climate, food, and debt risk.

In [ ]:
X_scaled = scaler.fit_transform(master_df[['rain_anomaly', 'food_volatility', 'debt_growth']])
master_df['risk_cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

plt.figure(figsize=(14, 6))
plt.scatter(master_df['year'], master_df['debt_growth'], c=master_df['risk_cluster'], s=200, cmap='coolwarm', edgecolors='black')
plt.plot(master_df['year'], master_df['debt_growth'], alpha=0.5, color='gray')
plt.title('Risk Eras: Debt Growth Segmented by Multi-Pillar Risk Clusters')
plt.xlabel('Year')
plt.ylabel('Debt Growth Rate')
plt.show()